# UAS KOMPILASI

#### Nama      : Reyhan Mahendra
#### Nim       : 221011450264
#### Kelas     : 06TPLE003   

#### 1. Konstruksi Sintaksis & Tata Bahasa (Grammar)
Konstruksi yang dipilih untuk proyek ini adalah Perulangan (Looping) while. Aturan sintaksis didefinisikan menggunakan pendekatan Backus-Naur Form (BNF) sebagai berikut:

In [ ]:

```text
 ::= "while" "("  ")" "{"  "}"
  ::=   
 ::=  "=" 
      ::= "<" | ">" | "==" | "!=" | "<=" | ">="
     

#### Kode Program Mini-Compiler

In [ ]:
import re

class WhileCompiler:
    def __init__(self, source_code):
        self.source_code = source_code
        self.label_counter = 1
        
    def new_label(self):
        lbl = f"L{self.label_counter}"
        self.label_counter += 1
        return lbl

    def lexical_analysis(self):
        # 1. TAHAPAN LEKSIKAL
        spaced_code = re.sub(r'([(){}=+\-*/<>!]+)', r' \1 ', self.source_code)
        tokens = spaced_code.split()
        return tokens

    def syntax_semantic_analysis(self, tokens):
        # 2 & 3. TAHAPAN SINTAKSIS & SEMANTIK
        if tokens[0] != 'while':
            raise SyntaxError("Error Sintaksis: Harus diawali dengan kata kunci 'while'.")
        if tokens[1] != '(':
            raise SyntaxError("Error Sintaksis: Kriteria kondisi harus dibuka dengan '('.")
        
        try:
            close_paren_idx = tokens.index(')')
        except ValueError:
            raise SyntaxError("Error Sintaksis: Kurang tanda penutup kondisi ')'.")
            
        condition_tokens = tokens[2:close_paren_idx]
        condition_str = " ".join(condition_tokens)
        
        operators = ['<', '>', '==', '!=', '<=', '>=']
        if not any(op in condition_tokens for op in operators):
            raise TypeError("Error Semantik: Kondisi perulangan tidak memiliki operator relasi yang valid.")
            
        if tokens[close_paren_idx + 1] != '{' or tokens[-1] != '}':
            raise SyntaxError("Error Sintaksis: Blok instruksi harus diapit oleh '{' dan '}'.")
            
        body_tokens = tokens[close_paren_idx + 2 : -1]
        body_str = " ".join(body_tokens)
        
        return condition_str, body_str

    def generate_tac(self):
        # 4. TAHAPAN GENERASI KODE (Three-Address Code)
        tokens = self.lexical_analysis()
        condition, body = self.syntax_semantic_analysis(tokens)
        
        label_start = self.new_label()
        label_end = self.new_label()
        
        tac = []
        tac.append(f"{label_start}:")
        tac.append(f"ifFalse {condition} goto {label_end}")
        tac.append(f"    {body}")
        tac.append(f"    goto {label_start}")
        tac.append(f"{label_end}:")
        
        return "\n".join(tac)

#### Pengujian Kasus

In [ ]:
source_input = "while ( x < 10 ) { x = x + 1 }"
compiler = WhileCompiler(source_input)

print("=== 1. HASIL ANALISIS LEKSIKAL (TOKENS) ===")
tokens_result = compiler.lexical_analysis()
print(tokens_result)
print("\n" + "="*40)

print("=== 2 & 3. ANALISIS SINTAKSIS & SEMANTIK ===")
cond, body = compiler.syntax_semantic_analysis(tokens_result)
print(f"Kondisi Valid : {cond}")
print(f"Body Statement : {body}")
print("\n" + "="*40)

print("=== 4. GENERASI THREE-ADDRESS CODE (TAC) ===")
print(compiler.generate_tac())